# COMEX Cash-and-Carry Arbitrage

Physical gold (GC) and silver (SI) rolling carry strategy.

| Parameter | Value |
|-----------|-------|
| Metals | GC (Feb/Apr/Jun/Aug/Oct/Dec), SI (Jan/Mar/May/Jul/Sep/Dec) |
| Physical hold | FDD1 → FDD2 (first business day of delivery month, ~60-day hold per leg) |
| Carry | Switch − Funding (SOFR OIS, 15-pillar, fixed via IRS at entry) − Storage |
| Entry | `carry_bp_ann ≥ CARRY_MIN_BP` — borrow cash, buy F1, sell F2 |
| At each FDD2 | **Roll** (sell new F2, keep physical) if carry ≥ threshold; else **deliver and close** |
| Direction | **LONG ONLY** — no stop-loss, no signal exit |
| Expected return | ~100 bp p.a. from carry accrual net of funding and storage |

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path
from itertools import product
from typing import Optional

# ── CONFIG ───────────────────────────────────────────────────────────────────
CONFIG = {
    # Data
    'DATA_PATH':         Path('data/data.csv'),
    'OUT_DIR':           Path('outputs_comex'),

    # Contract months (bi-monthly for GC matches active delivery cycle)
    'GC_MONTHS':         [2, 4, 6, 8, 10, 12],     # Feb Apr Jun Aug Oct Dec
    'SI_MONTHS':         [1, 3, 5, 7, 9, 12],       # Jan Mar May Jul Sep Dec

    # Physical storage cost (bp/pa)
    'GC_STORAGE_BPPA':   15.0,
    'SI_STORAGE_BPPA':   25.0,

    # 15-pillar SOFR OIS curve  (column_name, tenor_days)
    'SOFR_PILLARS': [
        ('sofr_ois_1w',   7), ('sofr_ois_2w',  14),
        ('sofr_ois_1m',  30), ('sofr_ois_2m',  61), ('sofr_ois_3m',  91),
        ('sofr_ois_4m', 122), ('sofr_ois_5m', 152), ('sofr_ois_6m', 182),
        ('sofr_ois_7m', 213), ('sofr_ois_8m', 243), ('sofr_ois_9m', 273),
        ('sofr_ois_10m',304), ('sofr_ois_11m',334),
        ('sofr_ois_1y', 365), ('sofr_ois_2y', 730),
    ],

    # Entry signal — minimum net carry (bp ann, net of funding + storage)
    # No z-score: enter whenever carry clears this absolute hurdle
    'CARRY_MIN_BP':      100.0,

    # Execution cost per leg (bp); round-trip = 2×
    'EXEC_COST_BP':      0.50,

    # Block entry within this many days of FDD1
    'MIN_DTE_ENTRY':     5,
}
CONFIG['OUT_DIR'].mkdir(exist_ok=True)
print('CONFIG loaded.')
print(f"  SOFR pillars  : {len(CONFIG['SOFR_PILLARS'])}")
print(f"  GC months     : {CONFIG['GC_MONTHS']}")
print(f"  SI months     : {CONFIG['SI_MONTHS']}")
print(f"  Carry min (bp): {CONFIG['CARRY_MIN_BP']}")

## Section 1 — Data Load

Load `data/data.csv`, parse dates, validate required columns.

In [ ]:
df = pd.read_csv(CONFIG['DATA_PATH'], parse_dates=['date'], index_col='date')
df.sort_index(inplace=True)

# Validate key columns
REQ = ['gc_fut_front', 'gc_fut_second', 'si_fut_front', 'si_fut_second', 'fed_funds_target']
missing = [c for c in REQ if c not in df.columns]
if missing:
    raise ValueError(f'Missing columns: {missing}')

print(f'Loaded {len(df):,} rows  {df.index[0].date()} → {df.index[-1].date()}')
print(f'Total columns: {len(df.columns)}')
print()

# Coverage audit
for col in ['gc_fut_front', 'gc_fut_second', 'si_fut_front', 'si_fut_second']:
    n = df[col].notna().sum()
    print(f'  {col:20s}  {n:5d} rows  ({n/len(df)*100:.0f}%  non-NaN)')

# SOFR pillar availability
available_pillars = [(c, d) for c, d in CONFIG['SOFR_PILLARS'] if c in df.columns]
print(f'\nSOFR pillars in data: {len(available_pillars)} / {len(CONFIG["SOFR_PILLARS"])}')
print('  ' + ', '.join(c for c, _ in available_pillars))

## Section 2 — FDD Calendar

Compute **First Delivery Day** for each contract month:
- **GC**: Feb / Apr / Jun / Aug / Oct / Dec (active bi-monthly delivery cycle, ~60-day hold)
- **SI**: Jan / Mar / May / Jul / Sep / Dec (COMEX silver cycle)

**FDD = first business day of the delivery month** (Mon–Fri, weekday roll-forward only).

For each trading date we find the *current front* and *second* FDD, then compute:
- `dte_fdd_gc1 / dte_fdd_si1` — calendar days to FDD of front contract
- `dte_fdd_gc2 / dte_fdd_si2` — calendar days to FDD of second contract
- `day_count_gc / day_count_si` — `dte2 − dte1` (physical hold period, target ~60 days)

In [ ]:
def first_biz_day(year: int, month: int) -> pd.Timestamp:
    """Return first Mon-Fri day of given year/month (no holiday calendar)."""
    d = np.datetime64(f'{year:04d}-{month:02d}-01', 'D')
    return pd.Timestamp(np.busday_offset(d, 0, roll='forward'))


def build_fdd_list(months: list, years) -> list:
    """Sorted list of FDD timestamps for given months across year range."""
    return sorted(first_biz_day(y, m) for y in years for m in months)


def compute_dte_pair(
    dates: pd.DatetimeIndex,
    fdd_list: list,
) -> tuple:
    """
    For each date, find next FDD1 (>= date) and the subsequent FDD2.
    Returns (dte1_arr, dte2_arr) in calendar days.
    """
    fdd_ns = np.array([f.value for f in fdd_list], dtype='int64')
    date_ns = dates.astype('int64').values

    dte1 = np.full(len(dates), np.nan)
    dte2 = np.full(len(dates), np.nan)

    for i, (d, dv) in enumerate(zip(dates, date_ns)):
        idx = int(np.searchsorted(fdd_ns, dv, side='left'))
        if idx >= len(fdd_list) - 1:
            continue
        dte1[i] = (fdd_list[idx]     - d).days
        dte2[i] = (fdd_list[idx + 1] - d).days

    return dte1, dte2


# Build FDD lists spanning data range +/- 2 years
years = range(df.index[0].year - 1, df.index[-1].year + 3)
gc_fdds = build_fdd_list(CONFIG['GC_MONTHS'], years)
si_fdds = build_fdd_list(CONFIG['SI_MONTHS'], years)

gc_dte1, gc_dte2 = compute_dte_pair(df.index, gc_fdds)
si_dte1, si_dte2 = compute_dte_pair(df.index, si_fdds)

df['dte_fdd_gc1'] = gc_dte1
df['dte_fdd_gc2'] = gc_dte2
df['day_count_gc'] = df['dte_fdd_gc2'] - df['dte_fdd_gc1']

df['dte_fdd_si1'] = si_dte1
df['dte_fdd_si2'] = si_dte2
df['day_count_si'] = df['dte_fdd_si2'] - df['dte_fdd_si1']

print('FDD calendar computed.')
n_gc_in_range = sum(1 for f in gc_fdds if df.index[0] <= f <= df.index[-1])
n_si_in_range = sum(1 for f in si_fdds if df.index[0] <= f <= df.index[-1])
print(f'  GC FDDs in data range: {n_gc_in_range}')
print(f'  SI FDDs in data range: {n_si_in_range}')
print()

cols = ['dte_fdd_gc1', 'dte_fdd_gc2', 'day_count_gc',
        'dte_fdd_si1', 'dte_fdd_si2', 'day_count_si']
sample = df[cols].dropna()
print(f'Valid rows: {len(sample)}')
print(sample.describe().round(1))

## Section 3 — SOFR OIS Curve Interpolation (15 Pillars)

Build a daily SOFR OIS rate interpolated to the **physical hold tenor** (`day_count = dte_fdd2 − dte_fdd1`).

| Method | Detail |
|--------|--------|
| Interpolation | `numpy.interp` — piecewise linear between active pillars |
| Extrapolation | Flat (hold first/last pillar value) |
| Fallback | `fed_funds_target` when fewer than 2 pillars have data |
| Pillars | 1W 2W 1M 2M 3M 4M 5M 6M 7M 8M 9M 10M 11M 1Y 2Y |

In [ ]:
# Build active pillar arrays (columns present in data)
PILLAR_COLS = [c for c, _ in CONFIG['SOFR_PILLARS'] if c in df.columns]
PILLAR_DAYS = np.array([d for c, d in CONFIG['SOFR_PILLARS'] if c in df.columns])
print(f'Active SOFR pillars: {len(PILLAR_COLS)}')
for col, days in zip(PILLAR_COLS, PILLAR_DAYS):
    cov = df[col].notna().sum()
    print(f'  {col:20s}  {days:4d}d   {cov:5d} rows  ({cov/len(df)*100:.0f}%)')


def sofr_interp_vec(
    sofr_mat: np.ndarray,       # (N, n_pillars)
    pillar_days: np.ndarray,    # (n_pillars,)
    tenor_arr: np.ndarray,      # (N,)
    fallback_arr: np.ndarray,   # (N,)
) -> np.ndarray:
    """
    Row-by-row linear SOFR interpolation using numpy.interp.
    numpy.interp gives flat extrapolation at endpoints.
    Falls back to fed_funds_target when < 2 valid pillars.
    """
    results = np.full(len(tenor_arr), np.nan)
    for i in range(len(tenor_arr)):
        row = sofr_mat[i]
        valid = ~np.isnan(row)
        if valid.sum() < 2:
            results[i] = fallback_arr[i]  # fed_funds_target fallback
            continue
        xs = pillar_days[valid]
        ys = row[valid]
        tenor = tenor_arr[i]
        if np.isnan(tenor):
            continue
        results[i] = float(np.interp(tenor, xs, ys))
    return results


sofr_mat     = df[PILLAR_COLS].values
fallback_arr = df['fed_funds_target'].values if 'fed_funds_target' in df.columns \
               else np.full(len(df), np.nan)

df['sofr_interp_gc'] = sofr_interp_vec(
    sofr_mat, PILLAR_DAYS,
    df['day_count_gc'].values, fallback_arr
)
df.loc[df['day_count_gc'].isna(), 'sofr_interp_gc'] = np.nan

df['sofr_interp_si'] = sofr_interp_vec(
    sofr_mat, PILLAR_DAYS,
    df['day_count_si'].values, fallback_arr
)
df.loc[df['day_count_si'].isna(), 'sofr_interp_si'] = np.nan

print('\nSOFR interpolation complete.')
for col in ['sofr_interp_gc', 'sofr_interp_si']:
    s = df[col].dropna()
    print(f'  {col}:  mean={s.mean():.2f}%  min={s.min():.2f}%  max={s.max():.2f}%  n={len(s)}')

## Section 4 — Carry Calculation

**Cash-and-carry carry decomposition** (all in USD, then annualised to bp):

| Component | Formula |
|-----------|---------|
| Switch | `F2 − F1` (USD/oz) |
| Funding | `(SOFR_interp / 100) × t × F1` |
| Storage | `(storage_bppa / 10000) × t × F1` |
| **Net Carry** | `Switch − Funding − Storage` (USD) |
| Carry bp ann | `carry_usd / F1 / t × 10 000` |

Where `t = day_count / 360`.  
Positive carry ⟹ futures premium exceeds financing + storage cost.

In [ ]:
def compute_carry(df: pd.DataFrame, metal: str, storage_bppa: float) -> pd.DataFrame:
    """
    Compute carry decomposition for a metal.
    Adds switch_bp_ann, funding_bp_ann, storage_bp_ann, carry_bp_ann columns.
    """
    f1   = df[f'{metal}_fut_front'].astype(float)
    f2   = df[f'{metal}_fut_second'].astype(float)
    sofr = df[f'sofr_interp_{metal}'].astype(float)
    dc   = df[f'day_count_{metal}'].astype(float)

    t            = dc / 360.0
    switch_usd   = f2 - f1
    funding_usd  = (sofr / 100.0) * t * f1
    storage_usd  = (storage_bppa / 10000.0) * t * f1
    carry_usd    = switch_usd - funding_usd - storage_usd

    with np.errstate(divide='ignore', invalid='ignore'):
        scale = np.where((t > 0) & f1.notna() & (f1 > 0), 10000.0 / (f1 * t), np.nan)

    df[f'switch_usd_{metal}']    = switch_usd
    df[f'carry_usd_{metal}']     = carry_usd
    df[f'switch_bp_ann_{metal}'] = switch_usd   * scale
    df[f'funding_bp_ann_{metal}']= funding_usd  * scale
    df[f'storage_bp_ann_{metal}']= storage_usd  * scale
    df[f'carry_bp_ann_{metal}']  = carry_usd    * scale
    return df


df = compute_carry(df, 'gc', CONFIG['GC_STORAGE_BPPA'])
df = compute_carry(df, 'si', CONFIG['SI_STORAGE_BPPA'])

print('Carry computed.')
for metal, label in [('gc', 'Gold GC'), ('si', 'Silver SI')]:
    sw  = df[f'switch_bp_ann_{metal}'].dropna()
    fu  = df[f'funding_bp_ann_{metal}'].dropna()
    st  = df[f'storage_bp_ann_{metal}'].dropna()
    net = df[f'carry_bp_ann_{metal}'].dropna()
    print(f'\n{label}:')
    print(f'  Switch   : {sw.mean():+7.1f} bp  ({sw.min():.0f} – {sw.max():.0f})')
    print(f'  Funding  : {fu.mean():+7.1f} bp')
    print(f'  Storage  : {st.mean():+7.1f} bp')
    print(f'  Net carry: {net.mean():+7.1f} bp  ({net.min():.0f} – {net.max():.0f})')
    print(f'  % carry>0: {(net > 0).mean()*100:.0f}%')

## Section 5 — Entry Signal: Absolute Carry Threshold

Enter whenever `carry_bp_ann ≥ CARRY_MIN_BP` (default **100 bp**).

`carry_bp_ann` is already net of SOFR funding and storage — so 100 bp is the minimum net return  
required before execution costs to initiate a trade.

No z-score, no rolling lookback, no regime conditioning: if the arb clears the hurdle, do it.

In [ ]:
THR = CONFIG['CARRY_MIN_BP']

for metal, label in [('gc', 'Gold GC'), ('si', 'Silver SI')]:
    carry = df[f'carry_bp_ann_{metal}'].dropna()
    above = (carry >= THR).mean() * 100
    below = (carry < THR).mean()  * 100
    print(f'{label}:')
    print(f'  carry_bp_ann >= {THR:.0f} bp : {above:.0f}% of days  →  tradeable')
    print(f'  carry_bp_ann <  {THR:.0f} bp : {below:.0f}% of days  →  no trade')
    print(f'  mean carry when above: {carry[carry >= THR].mean():.0f} bp')
    print(f'  mean carry when below: {carry[carry <  THR].mean():.0f} bp')
    print()

## Section 6 — Carry History

Two-panel chart: GC and SI `carry_bp_ann` over time with the **100 bp entry threshold** marked.  
Shaded regions show periods where carry clears the hurdle and a new trade could be initiated.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=True)
fig.suptitle('COMEX Cash-and-Carry — Net Carry (bp ann)', fontsize=14, fontweight='bold')

THR = CONFIG['CARRY_MIN_BP']

for ax, (metal, label, color) in zip(axes, [
    ('gc', 'Gold (GC)', '#1565C0'),
    ('si', 'Silver (SI)', '#BF360C'),
]):
    carry = df[f'carry_bp_ann_{metal}'].dropna()

    ax.plot(carry.index, carry.values, lw=0.9, color=color, alpha=0.85, label='carry bp ann')
    ax.axhline(THR, color='green', lw=1.4, ls='--', label=f'entry threshold {THR:.0f} bp')
    ax.axhline(0,   color='black', lw=0.5, ls='--')

    # Shade tradeable periods (carry >= threshold)
    ax.fill_between(carry.index, carry.values, THR,
                    where=(carry.values >= THR),
                    alpha=0.15, color='green', label='tradeable (carry ≥ threshold)')

    ax.set_title(label, fontsize=11)
    ax.set_ylabel('bp ann')
    ax.legend(fontsize=8, loc='upper left')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.grid(alpha=0.3)

plt.tight_layout()
out = CONFIG['OUT_DIR'] / 'cc_01_carry_history.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

## Section 7 — Carry Attribution Decomposition

Stacked area chart showing how **Switch**, **−Funding**, and **−Storage** each contribute to net carry (bp ann) over time.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Carry Attribution: Switch vs Funding vs Storage', fontsize=13, fontweight='bold')

for ax, (metal, label) in zip(axes, [('gc', 'Gold (GC)'), ('si', 'Silver (SI)')]):
    cols = [f'switch_bp_ann_{metal}', f'funding_bp_ann_{metal}',
            f'storage_bp_ann_{metal}', f'carry_bp_ann_{metal}']
    sub = df[cols].dropna()

    ax.stackplot(
        sub.index,
        sub[f'switch_bp_ann_{metal}'],
        -sub[f'funding_bp_ann_{metal}'],
        -sub[f'storage_bp_ann_{metal}'],
        labels=['Switch', '−Funding', '−Storage'],
        colors=['#2196F3', '#F44336', '#FF9800'],
        alpha=0.7,
    )
    ax.plot(sub.index, sub[f'carry_bp_ann_{metal}'],
            color='black', lw=1.4, label='Net Carry', zorder=5)
    ax.axhline(0, color='black', lw=0.5, ls='--')
    ax.set_title(label, fontsize=11)
    ax.set_ylabel('bp ann')
    ax.legend(fontsize=8, loc='upper left')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.grid(alpha=0.3)

plt.tight_layout()
out = CONFIG['OUT_DIR'] / 'cc_02_attribution_decomp.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

## Section 8 — Backtest: LONG-Only Rolling Cash-and-Carry

**Trade structure (per leg):**
- **Entry / Roll**: `carry_bp_ann ≥ CARRY_MIN_BP` AND `dte_fdd1 > MIN_DTE_ENTRY`
  - Borrow cash at SOFR (fixed via simultaneous IRS — no interest-rate risk after entry)
  - Buy F1 (take physical delivery at FDD1), sell F2 (make delivery at FDD2)
- **FDD1**: Receive physical metal; carry begins accruing
- **FDD2**: Binary decision only:
  - **Roll** → sell new next contract, keep physical, lock new SOFR via IRS
  - **Close** → make physical delivery, repay loan, unwind IRS; arb complete

**No stop-loss. No signal exit. The only exit is when carry no longer clears the hurdle at a roll date.**

**P&L (closed arb — no basis or MTM risk):**
```
carry_pnl_bp  = locked_carry_bp_ann × day_count / 360   (fully determined at leg entry)
total_pnl_bp  = carry_pnl_bp − exec_cost_bp × 2
```

IRS note: the SOFR OIS rate at each leg entry is the fixed rate paid on the interest-rate swap.
Because carry is locked in full at entry, there is zero interest-rate risk over the physical hold.

In [ ]:
MONTH_ABB = {1:'Jan', 2:'Feb', 3:'Mar', 4:'Apr', 5:'May', 6:'Jun',
             7:'Jul', 8:'Aug', 9:'Sep', 10:'Oct', 11:'Nov', 12:'Dec'}


def backtest_cc(
    df: pd.DataFrame,
    metal: str = 'gc',
    carry_min_bp: float = 100.0,
    execcostbp: float = 0.50,
    iscutoff: Optional[pd.Timestamp] = None,
) -> tuple:
    """
    LONG-only rolling physical cash-and-carry backtest.

    Leg lifecycle:
      Enter  : carry_bp_ann >= carry_min_bp AND dte_fdd1 > MIN_DTE_ENTRY
               → borrow cash (SOFR fixed via IRS), buy F1, sell F2
      FDD1   : take physical delivery; carry accrues daily
      FDD2   : roll if carry >= carry_min_bp, else deliver and close

    P&L per leg (no basis / MTM risk — closed arb):
      carry_pnl  = locked_carry_bp_ann × day_count / 360
      total_pnl  = carry_pnl − exec_cost × 2

    Exec cost charged at every leg boundary (entry, each roll, final close).
    Roll: exit old leg (−exec) + enter new leg (−exec) on same day.
    """
    f1_col    = f'{metal}_fut_front'
    f2_col    = f'{metal}_fut_second'
    carry_col = f'carry_bp_ann_{metal}'
    dte1_col  = f'dte_fdd_{metal}1'
    dte2_col  = f'dte_fdd_{metal}2'
    dc_col    = f'day_count_{metal}'

    data = df.copy()
    if iscutoff is not None:
        data = data[data.index <= pd.Timestamp(iscutoff)]

    daily_pnl = pd.Series(0.0, index=data.index, dtype=float)
    trades: list = []

    in_trade       = False
    entry_date     = None   # start of this leg
    leg_f1         = None
    leg_f2         = None
    leg_switch     = None
    leg_carry      = None   # carry locked for this leg (bp ann)
    leg_day_count  = None   # physical hold days (day_count = dte2 - dte1)
    leg_dte        = None
    fdd1_date      = None   # take delivery
    fdd2_date      = None   # make delivery / roll decision

    min_dte = CONFIG['MIN_DTE_ENTRY']

    for date, row in data.iterrows():
        carry = row[carry_col]
        f1    = row[f1_col]
        f2    = row[f2_col]
        dte1  = row[dte1_col]
        dte2  = row[dte2_col]
        dc    = row[dc_col]

        if any(pd.isna(v) for v in [carry, f1, f2, dte1, dte2, dc]):
            continue

        if in_trade:
            # ── Carry accrues during physical hold: FDD1 ≤ date < FDD2 ──
            if fdd1_date <= date < fdd2_date:
                daily_pnl.loc[date] += leg_carry / 360.0

            # ── At FDD2: book leg, then check roll ──
            if date >= fdd2_date:
                carry_pnl = leg_carry / 360.0 * leg_day_count
                total_pnl = carry_pnl - execcostbp * 2.0

                f1_lbl = f"{metal.upper()} {MONTH_ABB[fdd1_date.month]}-{fdd1_date.year}"
                f2_lbl = f"{metal.upper()} {MONTH_ABB[fdd2_date.month]}-{fdd2_date.year}"

                trades.append({
                    'entrydate':        entry_date,
                    'fdd1date':         fdd1_date,
                    'exitdate':         date,
                    'metal':            metal.upper(),
                    'f1contract':       f1_lbl,
                    'f2contract':       f2_lbl,
                    'f1entry':          round(leg_f1,     2),
                    'f2entry':          round(leg_f2,     2),
                    'switchentry':      round(leg_switch, 2),
                    'entrycarry':       round(leg_carry,  2),
                    'daycount':         int(leg_day_count),
                    'holddays':         (date - entry_date).days,
                    'entrydtefdd1':     int(leg_dte),
                    'carrypnlbp':       round(carry_pnl,  2),
                    'execcostbp_total': round(execcostbp * 2.0, 2),
                    'totalpnlbp':       round(total_pnl,  2),
                })

                daily_pnl.loc[date] -= execcostbp   # exit exec (one leg of the roll)
                in_trade = False

        # ── Entry or roll: check carry threshold ──
        if not in_trade and carry >= carry_min_bp and dte1 > min_dte:
            in_trade      = True
            entry_date    = date
            leg_f1        = f1
            leg_f2        = f2
            leg_switch    = f2 - f1
            leg_carry     = carry
            leg_day_count = dc
            leg_dte       = dte1
            fdd1_date     = date + pd.Timedelta(days=int(dte1))
            fdd2_date     = date + pd.Timedelta(days=int(dte2))
            daily_pnl.loc[date] -= execcostbp   # entry exec (other leg of the roll)

    trade_log = pd.DataFrame(trades)
    return daily_pnl, trade_log


In [ ]:
# ── Run backtest with default CONFIG params ──
daily_pnl_gc, tlog_gc = backtest_cc(
    df, metal='gc',
    carry_min_bp=CONFIG['CARRY_MIN_BP'],
    execcostbp=CONFIG['EXEC_COST_BP'],
)
daily_pnl_si, tlog_si = backtest_cc(
    df, metal='si',
    carry_min_bp=CONFIG['CARRY_MIN_BP'],
    execcostbp=CONFIG['EXEC_COST_BP'],
)


def summary_stats(pnl: pd.Series, label: str, tlog: pd.DataFrame) -> None:
    cum = pnl.cumsum()
    ar  = pnl.mean() * 252
    vol = pnl.std()  * np.sqrt(252)
    sr  = ar / vol if vol > 0 else np.nan
    dd  = (cum - cum.cummax()).min()
    n_legs = len(tlog)
    avg_carry = tlog['entrycarry'].mean() if n_legs else 0
    print(f'\n{label}')
    print(f'  Total legs   : {n_legs}')
    print(f'  Avg entry carry: {avg_carry:.0f} bp ann')
    print(f'  Total P&L    : {pnl.sum():+7.1f} bp')
    print(f'  Ann return   : {ar:+7.1f} bp/yr')
    print(f'  Sharpe       :  {sr:.2f}')
    print(f'  Max drawdown : {dd:7.1f} bp')


summary_stats(daily_pnl_gc, 'GC (Gold C&C)', tlog_gc)
summary_stats(daily_pnl_si, 'SI (Silver C&C)', tlog_si)

# ── Annual breakdown ──
for metal, tlog in [('GC', tlog_gc), ('SI', tlog_si)]:
    if tlog.empty:
        continue
    tlog['year'] = pd.to_datetime(tlog['fdd1date']).dt.year
    ann = tlog.groupby('year').agg(
        legs=('totalpnlbp', 'count'),
        carry_pnl=('carrypnlbp', 'sum'),
        total_pnl=('totalpnlbp', 'sum'),
        avg_carry=('entrycarry', 'mean'),
    ).round(1)
    print(f'\n{metal} — Annual P&L (bp):')
    print(ann.to_string())

# ── Detailed leg breakdown ──
PRICE_COLS = ['entrydate', 'fdd1date', 'exitdate', 'f1contract', 'f2contract',
              'f1entry', 'f2entry', 'switchentry', 'entrycarry',
              'daycount', 'holddays', 'carrypnlbp', 'execcostbp_total', 'totalpnlbp']

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 160)

print('\n── GC Leg Breakdown ──')
if len(tlog_gc):
    display(tlog_gc[PRICE_COLS])

print('\n── SI Leg Breakdown ──')
if len(tlog_si):
    display(tlog_si[PRICE_COLS])

# Save trade logs
tlog_gc.to_csv(CONFIG['OUT_DIR'] / 'cc_trades_gc.csv', index=False)
tlog_si.to_csv(CONFIG['OUT_DIR'] / 'cc_trades_si.csv', index=False)
print('\nTrade logs saved.')


## Section 9 — Parameter Sweep

Sweep over `carry_min_bp` threshold values (5 levels × 2 metals = 10 backtests).

A higher threshold means fewer but higher-quality entries.  
Metrics: total P&L, annualised Sharpe, max drawdown, trade count.

In [ ]:
SWEEP_THRESHOLDS = [50.0, 75.0, 100.0, 125.0, 150.0]


def quick_stats(pnl: pd.Series) -> dict:
    cum = pnl.cumsum()
    ar  = pnl.mean() * 252
    vol = pnl.std()  * np.sqrt(252)
    sr  = ar / vol if vol > 0 else np.nan
    dd  = (cum - cum.cummax()).min()
    return {'ann_ret': ar, 'sharpe': sr, 'max_dd': dd}


print(f'Running {len(SWEEP_THRESHOLDS) * 2} backtests ({len(SWEEP_THRESHOLDS)} thresholds × 2 metals)...')

rows = []
for thr in SWEEP_THRESHOLDS:
    for metal in ('gc', 'si'):
        pnl, tlog = backtest_cc(
            df, metal=metal, carry_min_bp=thr,
            execcostbp=CONFIG['EXEC_COST_BP'],
        )
        st = quick_stats(pnl)
        rows.append({
            'metal':         metal.upper(),
            'carry_min_bp':  thr,
            'total_pnl':     round(pnl.sum(),     1),
            'ann_ret':       round(st['ann_ret'], 1),
            'sharpe':        round(st['sharpe'],  2),
            'max_dd':        round(st['max_dd'],  1),
            'n_trades':      len(tlog),
        })

sweep_df = pd.DataFrame(rows)
sweep_df.to_csv(CONFIG['OUT_DIR'] / 'cc_sweep_results.csv', index=False)

print('\nAll results — GC:')
display(sweep_df[sweep_df.metal == 'GC'].sort_values('carry_min_bp'))
print('\nAll results — SI:')
display(sweep_df[sweep_df.metal == 'SI'].sort_values('carry_min_bp'))

# ── Chart: Sharpe & trade count vs carry threshold ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Parameter Sweep — Sharpe & Trade Count vs Carry Threshold', fontsize=12)

for ax, metal in zip(axes, ['GC', 'SI']):
    sub = sweep_df[sweep_df.metal == metal].sort_values('carry_min_bp')

    color_sharpe = '#1565C0' if metal == 'GC' else '#BF360C'
    ax.bar(sub['carry_min_bp'], sub['sharpe'], width=12,
           color=color_sharpe, alpha=0.7, label='Sharpe')
    ax.axhline(0, color='black', lw=0.5)

    ax2 = ax.twinx()
    ax2.plot(sub['carry_min_bp'], sub['n_trades'], 'o--',
             color='grey', ms=6, lw=1.2, label='# trades')
    ax2.set_ylabel('# Trades', color='grey')
    ax2.tick_params(axis='y', labelcolor='grey')

    # Annotate default
    default_row = sub[sub['carry_min_bp'] == CONFIG['CARRY_MIN_BP']]
    if not default_row.empty:
        x = default_row['carry_min_bp'].values[0]
        ax.axvline(x, color='green', lw=1.2, ls='--', label=f'default {x:.0f}bp')

    ax.set_xlabel('carry_min_bp (threshold)')
    ax.set_ylabel('Sharpe')
    ax.set_title(f'{metal}')
    ax.legend(loc='upper left', fontsize=8)
    ax.grid(alpha=0.3)

plt.tight_layout()
out = CONFIG['OUT_DIR'] / 'cc_03_sweep.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')


## Section 10 — P&L Attribution per Trade

Decompose total P&L per trade into three components:

| Component | Description |
|-----------|-------------|
| **Carry P&L** | Locked carry accrual over hold period (`entry_carry / 360 × hold_days`) |
| **Basis P&L** | Actual spread move minus theoretical carry — measures basis convergence / divergence |
| **Exec Cost** | Round-trip commission (2 × exec_cost_bp) |

`total_pnl = carry_pnl + basis_pnl − exec_cost`

In [ ]:
tlog_all = pd.concat(
    [tlog_gc.assign(metal='GC'), tlog_si.assign(metal='SI')],
    ignore_index=True,
)

if tlog_all.empty:
    print('No legs to attribute.')
else:
    # ── Summary by metal and year ──
    tlog_all['year'] = pd.to_datetime(tlog_all['fdd1date']).dt.year
    summary = (
        tlog_all
        .groupby(['metal', 'year'])
        .agg(
            legs         = ('totalpnlbp', 'count'),
            avg_carry_bp = ('entrycarry',  'mean'),
            carry_pnl_bp = ('carrypnlbp',  'sum'),
            total_pnl_bp = ('totalpnlbp',  'sum'),
            avg_hold     = ('daycount',    'mean'),
        )
        .round(1)
    )
    print('P&L summary by metal × year:')
    display(summary)

    # ── Bar chart: carry P&L vs exec cost per leg ──
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('P&L per Leg: Carry vs Exec Cost', fontsize=13)

    for ax, metal in zip(axes, ['GC', 'SI']):
        sub = tlog_all[tlog_all.metal == metal].reset_index(drop=True)
        if sub.empty:
            ax.set_title(f'{metal} — No legs')
            continue

        x = np.arange(len(sub))
        w = 0.35
        ax.bar(x - w/2, sub['carrypnlbp'],       w, label='Carry P&L', color='#4CAF50', alpha=0.8)
        ax.bar(x + w/2, -sub['execcostbp_total'], w, label='Exec Cost', color='#F44336', alpha=0.8)
        ax.plot(x, sub['totalpnlbp'], 'ko-', ms=4, lw=1, label='Net P&L', zorder=5)
        ax.axhline(0, color='black', lw=0.5)
        ax.set_title(f'{metal} — {len(sub)} legs')
        ax.set_xlabel('Leg #')
        ax.set_ylabel('bp')
        ax.legend(fontsize=8)
        ax.set_xticks(x)
        ax.set_xticklabels([f'L{i+1}' for i in range(len(sub))], fontsize=7)
        ax.grid(alpha=0.3, axis='y')

    plt.tight_layout()
    out = CONFIG['OUT_DIR'] / 'cc_04_attribution.png'
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out}')

    tlog_all.to_csv(CONFIG['OUT_DIR'] / 'cc_attribution.csv', index=False)


## Section 11 — Cumulative P&L & Rolling Sharpe

Combined GC + SI equal-weight daily P&L stream, cumulative curve, and 63-day rolling Sharpe.

In [ ]:
pnl_combined = (daily_pnl_gc + daily_pnl_si).fillna(0)
cum_gc       = daily_pnl_gc.cumsum()
cum_si       = daily_pnl_si.cumsum()
cum_all      = pnl_combined.cumsum()

ROLL = 63
roll_sharpe = (
    pnl_combined.rolling(ROLL).mean() /
    pnl_combined.rolling(ROLL).std()
) * np.sqrt(252)

fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(14, 9), sharex=True,
    gridspec_kw={'height_ratios': [2, 1]},
)
fig.suptitle('COMEX C&C — Cumulative P&L & Rolling Sharpe', fontsize=13, fontweight='bold')

ax1.plot(cum_gc.index,  cum_gc.values,  lw=1.2, color='#1565C0', label='GC',       alpha=0.8)
ax1.plot(cum_si.index,  cum_si.values,  lw=1.2, color='#BF360C', label='SI',       alpha=0.8)
ax1.plot(cum_all.index, cum_all.values, lw=2.0, color='black',   label='Combined', zorder=5)
ax1.fill_between(cum_all.index, cum_all, 0,
                 where=(cum_all >= 0), alpha=0.08, color='green')
ax1.fill_between(cum_all.index, cum_all, 0,
                 where=(cum_all  < 0), alpha=0.08, color='red')
ax1.axhline(0, color='grey', lw=0.5, ls='--')
ax1.set_ylabel('Cumulative P&L (bp)')
ax1.legend(loc='upper left', fontsize=9)
ax1.grid(alpha=0.3)

ax2.plot(roll_sharpe.index, roll_sharpe.values, lw=1.0, color='purple',
         label=f'{ROLL}d Rolling Sharpe')
ax2.axhline( 0, color='grey',  lw=0.5, ls='--')
ax2.axhline( 1, color='green', lw=0.8, ls=':')
ax2.axhline(-1, color='red',   lw=0.8, ls=':')
ax2.set_ylabel('Rolling Sharpe')
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

plt.tight_layout()
out = CONFIG['OUT_DIR'] / 'cc_05_cum_pnl.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

print('\nFull-period statistics:')
for lbl, pnl in [('GC', daily_pnl_gc), ('SI', daily_pnl_si), ('Combined', pnl_combined)]:
    cum = pnl.cumsum()
    ar  = pnl.mean() * 252
    vol = pnl.std()  * np.sqrt(252)
    sr  = ar / vol if vol > 0 else np.nan
    dd  = (cum - cum.cummax()).min()
    print(f'  {lbl:10s}  ann={ar:+6.1f}bp  vol={vol:5.1f}  Sharpe={sr:5.2f}  MaxDD={dd:7.1f}bp')

## Section 12 — Stress Test

Evaluate strategy performance during four stress windows within the data range:

| Window | Dates | Event |
|--------|-------|-------|
| COVID Crash | 2020-02-20 → 2020-04-30 | Basis blow-out, repo stress |
| Fed Hike Cycle | 2022-01-01 → 2022-12-31 | Fastest rate hike since 1980s |
| Banking Crisis | 2023-03-01 → 2023-06-30 | SVB / Credit Suisse stress |
| XAG Lease Blowout | 2025-10-15 → 2025-11-30 | Silver-specific lease spike |

For each window: per-metal P&L (bp), max drawdown (bp), trade count.

In [ ]:
STRESS_WINDOWS = [
    ('COVID Crash',        '2020-02-20', '2020-04-30'),
    ('Fed Hike Cycle',     '2022-01-01', '2022-12-31'),
    ('Banking Crisis',     '2023-03-01', '2023-06-30'),
    ('XAG Lease Blowout',  '2025-10-15', '2025-11-30'),
]

stress_rows = []
for win_name, start, end in STRESS_WINDOWS:
    s, e = pd.Timestamp(start), pd.Timestamp(end)
    for metal in ('gc', 'si'):
        pnl, tlog = backtest_cc(
            df, metal=metal,
            carry_min_bp=CONFIG['CARRY_MIN_BP'],
            execcostbp=CONFIG['EXEC_COST_BP'],
        )
        wpnl  = pnl.loc[s:e]
        cum_w = wpnl.cumsum()
        total = wpnl.sum()
        maxdd = (cum_w - cum_w.cummax()).min() if len(cum_w) > 0 else np.nan
        if len(tlog):
            n_tr = len(tlog[(tlog['entrydate'] >= s) & (tlog['entrydate'] <= e)])
        else:
            n_tr = 0
        stress_rows.append({
            'window':       win_name,
            'metal':        metal.upper(),
            'total_pnl_bp': round(total, 1),
            'max_dd_bp':    round(maxdd, 1),
            'n_trades':     n_tr,
        })

stress_df = pd.DataFrame(stress_rows)
print('Stress Test Results:')
pivot_stress = stress_df.pivot_table(
    index='window', columns='metal',
    values=['total_pnl_bp', 'max_dd_bp', 'n_trades'],
    aggfunc='first',
).round(1)
display(pivot_stress)

# ── Cumulative P&L within each stress window ──
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Stress Windows — Cumulative P&L', fontsize=13, fontweight='bold')

for ax, (win_name, start, end) in zip(axes.flatten(), STRESS_WINDOWS):
    s, e = pd.Timestamp(start), pd.Timestamp(end)
    for metal, color, lbl in [('gc', '#1565C0', 'GC'), ('si', '#BF360C', 'SI')]:
        pnl, _ = backtest_cc(
            df, metal=metal,
            carry_min_bp=CONFIG['CARRY_MIN_BP'],
            execcostbp=CONFIG['EXEC_COST_BP'],
        )
        cum_w = pnl.loc[s:e].cumsum()
        if len(cum_w) > 0:
            ax.plot(cum_w.index, cum_w.values, lw=1.5, color=color, label=lbl)
    ax.axhline(0, color='grey', lw=0.5, ls='--')
    ax.set_title(f'{win_name}\n{start}  →  {end}', fontsize=10)
    ax.set_ylabel('Cum P&L (bp)')
    ax.legend(fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.grid(alpha=0.3)

plt.tight_layout()
out = CONFIG['OUT_DIR'] / 'cc_06_stress_test.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

stress_df.to_csv(CONFIG['OUT_DIR'] / 'cc_stress_results.csv', index=False)
print('Done. All outputs saved to outputs_comex/.')
